# 02a. Genome mapping with STAR and featureCounts

本ノートブックは、定量化パターン1「reference genomeにmappingしてgene count matrixを作る」方法である。

**今どの解析段階にいるか**

FASTQ readをヒトGenome上の位置に対応づけ、その後、annotationのGTFを使ってgeneごとのread数を数える段階である。

**このノートブックの役割**

- STARでpaired-end FASTQをGRCh38 genomeへsplice-aware mappingする。
- 座標順にsort済みのBAMファイルとBAM indexをサンプルごとに作る。
- featureCountsでGTF annotationとBAMを重ね合わせ、DESeq2に渡せるgene count matrixを作る。

**入力**

- `metadata/samples.tsv`（各サンプルの `sample_id`, 群情報, R1/R2 FASTQパスを持つ表）
- paired-end FASTQファイル（`samples.tsv` の `fastq_1`, `fastq_2` が指すreadファイル）
- `reference/gencode_grch38/GRCh38.primary_assembly.genome.fa`（STAR mapping先のGRCh38 genome配列）
- `reference/gencode_grch38/gencode.v49.annotation.gtf`（gene/exonのgenome上の位置を持つannotation）
- `reference/gencode_grch38/star_index/`（STARが高速にmappingするためのgenome index）

**出力**

- `results/quant/genome_mapping/star/<sample_id>/<sample_id>.Aligned.sortedByCoord.out.bam`（readのgenome上の位置を座標順にsortしたBAM）
- `results/quant/genome_mapping/star/<sample_id>/<sample_id>.Aligned.sortedByCoord.out.bam.bai`（BAMを高速に参照するためのindex）
- `results/quant/genome_mapping/featurecounts/featureCounts_gene_counts.txt`（featureCountsの生出力）
- `results/counts/gene_counts_genome_mapping.tsv`（行=gene、列=sampleのread count matrix）

**次に進む条件**

- サンプルごとに、すべてのsorted BAMがある。
- featureCounts出力から作ったcount matrixの列名が `samples.tsv` の `sample_id` と一致している。


### featureCounts コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `-T N` | 並列スレッド数。 |
| `-p` | ペアエンド（paired-end）モード。R1とR2のペアをまとめて1 fragmentとして集計する。 |
| `-s N` | Strand specificity（鎖特異性）。`0`=unstranded、`1`=stranded、`2`=reversely stranded。 |
| `-a FILE` | annotation GTFファイル（`gencode.v*.annotation.gtf`）を指定する。 |
| `-o FILE` | count結果の出力ファイルパスである。 |
| `BAM...` | STARが作ったサンプルごとのsorted BAMファイルを並べて渡す。 |

featureCountsは、STARが生成したBAMファイルを入力として、各readをGTFのexon annotationと照合し、geneごとのread countを集計する。出力は行=gene、列=sampleのcount matrixである。


In [ ]:
from pathlib import Path
import shlex
import shutil
import subprocess
import time

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

PROJECT_DIR = Path('/Users/yusuke_tateishi/Documents/RNA_seq').resolve()
SAMPLES = pd.read_csv(PROJECT_DIR / 'metadata/samples.tsv', sep='\t')

GENCODE_RELEASE = '49'
GENOME_FASTA_PATH = PROJECT_DIR / 'reference/gencode_grch38/GRCh38.primary_assembly.genome.fa'
GTF_PATH = PROJECT_DIR / 'reference/gencode_grch38/gencode.v49.annotation.gtf'
STAR_INDEX_DIR = PROJECT_DIR / 'reference/gencode_grch38/star_index'
STAR_ALIGN_DIR = PROJECT_DIR / 'results/quant/genome_mapping/star'
FEATURECOUNTS_DIR = PROJECT_DIR / 'results/quant/genome_mapping/featurecounts'
OUT_COUNT_MATRIX = PROJECT_DIR / 'results/counts/gene_counts_genome_mapping.tsv'

THREADS = 8
READ_LENGTH = 150
SJDB_OVERHANG = READ_LENGTH - 1
FEATURECOUNTS_STRANDNESS = 0
GUNZIP_PATH = shutil.which('gunzip') or ('/usr/bin/gunzip' if Path('/usr/bin/gunzip').exists() else None)
GZIP_PATH = shutil.which('gzip') or ('/usr/bin/gzip' if Path('/usr/bin/gzip').exists() else None)
if GUNZIP_PATH is not None:
    STAR_DECOMPRESSOR_COMMAND = [GUNZIP_PATH, '-c']
elif GZIP_PATH is not None:
    STAR_DECOMPRESSOR_COMMAND = [GZIP_PATH, '-cd']
else:
    STAR_DECOMPRESSOR_COMMAND = None
USE_BASH_PROCESS_SUBSTITUTION_FOR_STAR = True

SAMPLES[['sample_id', 'condition', 'fastq_1', 'fastq_2']]


## 解析ルートの全体像

パターン1では、readをまずgenomeへmappingする。その後、GTF annotationに書かれたexon領域を使ってgeneごとに数える。

```text
FASTQ
  -> STAR genome mapping
  -> sorted BAM
  -> featureCounts + GTF annotation
  -> gene count matrix
  -> DESeq2
```

genome mappingは、splice junctionをまたぐRNA-seq readも扱う。たとえばexon 1とexon 2にまたがるreadは、Genome上では途中にintronがあるため、通常のDNA read mappingよりRNA-seq専用の処理が必要である。

mappingの基本的な考え方は、read配列と参照配列の一致度を最大にする場所を探すことである。単純化すると、次のようなscoreを高くする位置を探する。

$$
\mathrm{score} = \mathrm{matches} - \mathrm{mismatches} - \mathrm{gap\ penalties}
$$

ここで `matches` はreadとgenomeが一致した塩基数、`mismatches` は違う塩基数である。STARはさらにsplice junction候補を使い、RNA-seq readらしい分割mappingを高速に探する。


## ツールが見えるか確認する

`STAR` はGenomemapping、`samtools` はBAM index作成、`featureCounts` はgeneごとのread countingに使う。

このノートブックでは、ツールが未導入でも最初から止まらないように、存在確認を表で表示する。


In [ ]:
tool_names = ['STAR', 'samtools', 'featureCounts', 'gzip']
tool_status = pd.DataFrame(
    [{'tool': name, 'path': shutil.which(name), 'available': shutil.which(name) is not None} for name in tool_names]
)
tool_status


## 参照ファイルが準備済みか確認する

この段階の「辞書」はtranscriptomeではなくgenomeである。

- genome FASTA: readをmappingする塩基配列本体。
- GTF: genome上のどの区間がどのgeneやexonかを示す注釈表。
- STAR index: STARが高速に検索するためにgenome FASTAから作る索引。

GTFの中身は次のような形である。

```text
chr1  HAVANA  exon  11121  11211  .  +  .  gene_id "ENSG..."; transcript_id "ENST..."; gene_name "DDX11L16";
```

これは「chr1の11121から11211が、あるtranscriptのexonである」という意味である。


In [ ]:
reference_status = pd.DataFrame(
    [
        {'name': 'genome_fasta', 'path': GENOME_FASTA_PATH.relative_to(PROJECT_DIR), 'exists': GENOME_FASTA_PATH.exists()},
        {'name': 'gtf_path', 'path': GTF_PATH.relative_to(PROJECT_DIR), 'exists': GTF_PATH.exists()},
        {'name': 'star_index_dir', 'path': STAR_INDEX_DIR.relative_to(PROJECT_DIR), 'exists': STAR_INDEX_DIR.exists()},
    ]
)
reference_status


## STAR index作成コマンドを作る

STAR indexは、genome FASTAを直接毎回探す代わりに、検索しやすい形へ変換したものである。

`SJDB_OVERHANG` はsplice junction databaseを作るときのread長に関係する値である。一般にread長が150 bpなら、

$$
\mathrm{sjdbOverhang} = \mathrm{read\ length} - 1 = 150 - 1 = 149
$$

とする。ここでは `READ_LENGTH = 150` と仮定している。実際のread長が違う場合は、このセルの `READ_LENGTH` を変更する。


### STAR index作成コマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `--runThreadN N` | STARが使う並列スレッド数。 |
| `--runMode genomeGenerate` | genome indexを作るモード。通常のalignmentでは指定しない。 |
| `--genomeDir DIR` | STAR indexの出力先ディレクトリ。 |
| `--genomeFastaFiles FILE` | mapping先になるGRCh38 genome FASTAを指定する。 |
| `--sjdbGTFfile FILE` | splice junction情報を作るためのGTF annotationを指定する。 |
| `--sjdbOverhang N` | read長から1を引いた値を指定する。150 bp readなら `149` が目安である。 |

このノートブックでは、`00b_reference_setup_gencode_grch38.ipynb` で作ったSTAR indexが既にあれば再作成しない。


In [ ]:
star_index_command = [
    'STAR',
    '--runThreadN', str(THREADS),
    '--runMode', 'genomeGenerate',
    '--genomeDir', str(STAR_INDEX_DIR),
    '--genomeFastaFiles', str(GENOME_FASTA_PATH),
    '--sjdbGTFfile', str(GTF_PATH),
    '--sjdbOverhang', str(SJDB_OVERHANG),
]

print(' '.join(star_index_command))


## STAR indexを作成・確認する

通常, STAR indexは [00b_reference_setup_gencode_grch38.ipynb](../00b_reference_setup_gencode_grch38.ipynb) ですでに作成されている。
すでに `star_index` ディレクトリ内にインデックスが存在する場合は構築を自動的にスキップする。未構築でここで新たに構築したい場合のみ `RUN_STAR_INDEX = True` に設定しよう。

> [!WARNING]
> ヒトGenome（GRCh38）に対するSTARのインデックス構築には、**通常30GB以上のRAM（メモリ）**が必要である。メモリ不足に注意しよう。


In [ ]:
"""%cd 
%cd /Users/yusuke_tateishi/Documents/RNA_seq/
!ls

!gzip -dk reference/gencode_grch38/GRCh38.primary_assembly.genome.fa.gz
!gzip -dk reference/gencode_grch38/gencode.v49.annotation.gtf.gz"""

In [ ]:
RUN_STAR_INDEX = False  # Keep False if already created in 00b

# すでに正常なSTAR indexファイルが存在するかチェック
is_indexed = (STAR_INDEX_DIR / 'Genome').exists() and (STAR_INDEX_DIR / 'SA').exists()

if is_indexed:
    print(f'STAR index already exists at: {STAR_INDEX_DIR}. Skipping build.')
elif RUN_STAR_INDEX:
    star = shutil.which('STAR')
    if star is None:
        raise RuntimeError('STAR was not found. Activate the rna-seq environment or install STAR first.')
    
    # genome_fastaとgtfが圧縮されている場合、一時的に解凍してSTARに渡す
    decompressed_fasta = GENOME_FASTA_PATH.with_suffix('') if GENOME_FASTA_PATH.suffix == '.gz' else GENOME_FASTA_PATH
    decompressed_gtf = GTF_PATH.with_suffix('') if GTF_PATH.suffix == '.gz' else GTF_PATH
    
    try:
        import gzip
        if GENOME_FASTA_PATH.suffix == '.gz' and not decompressed_fasta.exists():
            print(f'Decompressing {GENOME_FASTA_PATH.name}...')
            with gzip.open(GENOME_FASTA_PATH, 'rb') as f_in, decompressed_fasta.open('wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        if GTF_PATH.suffix == '.gz' and not decompressed_gtf.exists():
            print(f'Decompressing {GTF_PATH.name}...')
            with gzip.open(GTF_PATH, 'rb') as f_in, decompressed_gtf.open('wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
                
        STAR_INDEX_DIR.mkdir(parents=True, exist_ok=True)
        
        # コマンドの参照ファイルを一時的な非圧縮ファイルに書き換え
        command = [star] + star_index_command[1:]
        # パスを非圧縮のパスに変更
        for idx, arg in enumerate(command):
            if arg == str(GENOME_FASTA_PATH):
                command[idx] = str(decompressed_fasta)
            elif arg == str(GTF_PATH):
                command[idx] = str(decompressed_gtf)
                
        print('Running:', ' '.join(command))
        subprocess.run(command, check=True)
        print('STAR index built successfully.')
    finally:
        # クリーンアップ
        if GENOME_FASTA_PATH.suffix == '.gz' and decompressed_fasta.exists():
            print(f'Removing temporary decompressed FASTA: {decompressed_fasta}')
            decompressed_fasta.unlink()
        if GTF_PATH.suffix == '.gz' and decompressed_gtf.exists():
            print(f'Removing temporary decompressed GTF: {decompressed_gtf}')
            decompressed_gtf.unlink()
else:
    print('STAR index is missing, and RUN_STAR_INDEX is False.')
    print('Please set RUN_STAR_INDEX = True or run notebook 00b to prepare the index.')


## STAR alignmentコマンドを作る

STAR alignmentでは、各sampleのR1/R2 FASTQをgenome indexへmappingし、座標順にsort済みのBAMを作る。

BAMは「どのreadがgenomeのどこにmappingされたか」を持つファイルである。次のfeatureCountsでは、このBAMとGTFを重ね合わせる。


### STAR alignmentコマンドライン引数の説明

| 引数 | 意味 |
|------|------|
| `--runThreadN N` | STARが使う並列スレッド数。 |
| `--genomeDir DIR` | 事前に作成したSTAR genome indexを指定する。 |
| `--readFilesIn R1 R2` | ペアエンドFASTQのR1/R2を指定する。この環境ではSTAR内部の `--readFilesCommand` が失敗するため、`bash` のprocess substitutionで `gunzip -c` の出力をSTARへ渡す。 |
| `--outSAMtype BAM SortedByCoordinate` | 出力をBAM形式にし、genome座標順にsortして保存する。 |
| `--outFileNamePrefix PREFIX` | STAR出力ファイル名のprefixを指定する。sampleごとに別ディレクトリへ出す。 |

この後の `samtools index BAM` は、作成したBAMに `.bai` indexを付ける処理である。featureCounts自体には必須ではないが、BAM確認や可視化で便利である。

`USE_BASH_PROCESS_SUBSTITUTION_FOR_STAR = True` のときは、STARのalignment条件は変えず、圧縮FASTQの展開だけを外側の `bash` に任せる。`--readFilesCommand` で問題なく動く環境では、この値を `False` にするとSTAR標準の圧縮入力方式に戻せる。


In [ ]:
if STAR_DECOMPRESSOR_COMMAND is None:
    raise RuntimeError('gunzip or gzip was not found. STAR needs a decompressor to read .fq.gz files.')

star_align_commands = []
bam_paths = []
star_tmp_paths = []

for _, sample in SAMPLES.iterrows():
    sample_id = sample['sample_id']
    sample_dir = STAR_ALIGN_DIR / sample_id
    out_prefix = sample_dir / f'{sample_id}.'
    bam_path = sample_dir / f'{sample_id}.Aligned.sortedByCoord.out.bam'
    star_tmp_path = Path(str(out_prefix) + '_STARtmp')
    fastq_1 = PROJECT_DIR / sample['fastq_1']
    fastq_2 = PROJECT_DIR / sample['fastq_2']

    if USE_BASH_PROCESS_SUBSTITUTION_FOR_STAR:
        command = (
            shlex.join([
                'STAR',
                '--runThreadN', str(THREADS),
                '--genomeDir', str(STAR_INDEX_DIR),
                '--readFilesIn',
            ])
            + f' <({shlex.join(STAR_DECOMPRESSOR_COMMAND + [str(fastq_1)])})'
            + f' <({shlex.join(STAR_DECOMPRESSOR_COMMAND + [str(fastq_2)])}) '
            + shlex.join([
                '--outSAMtype', 'BAM', 'SortedByCoordinate',
                '--outFileNamePrefix', str(out_prefix),
            ])
        )
    else:
        command = [
            'STAR',
            '--runThreadN', str(THREADS),
            '--genomeDir', str(STAR_INDEX_DIR),
            '--readFilesIn', str(fastq_1), str(fastq_2),
            '--readFilesCommand', *STAR_DECOMPRESSOR_COMMAND,
            '--outSAMtype', 'BAM', 'SortedByCoordinate',
            '--outFileNamePrefix', str(out_prefix),
        ]

    star_align_commands.append(command)
    bam_paths.append(bam_path)
    star_tmp_paths.append(star_tmp_path)

print('First STAR alignment command:')
first_command = star_align_commands[0]
print(first_command if isinstance(first_command, str) else ' '.join(first_command))
print()
print('Expected first BAM:')
print(bam_paths[0].relative_to(PROJECT_DIR))


## STAR alignmentを実行し、BAM indexを作る

初回だけ `RUN_STAR_ALIGN = True` にする。sampleごとにsorted BAMを作り、その後 `samtools index` で `.bai` indexを作る。

`.bai` はBAMを高速に参照するための索引である。可視化や一部の下流処理で必要になる。


In [ ]:
RUN_STAR_ALIGN = True
CLEAN_FAILED_STAR_OUTPUT = True

if RUN_STAR_ALIGN:
    star = shutil.which('STAR')
    samtools = shutil.which('samtools')
    if star is None:
        raise RuntimeError('STAR was not found. Activate the rna-seq environment or install STAR first.')
    if samtools is None:
        raise RuntimeError('samtools was not found. Activate the rna-seq environment first.')
    if not STAR_INDEX_DIR.exists():
        raise FileNotFoundError(STAR_INDEX_DIR)
    if STAR_DECOMPRESSOR_COMMAND is None:
        raise RuntimeError('gunzip or gzip was not found. STAR needs a decompressor to read .fq.gz files.')
    if USE_BASH_PROCESS_SUBSTITUTION_FOR_STAR and shutil.which('bash') is None and not Path('/bin/bash').exists():
        raise RuntimeError('bash was not found. Set USE_BASH_PROCESS_SUBSTITUTION_FOR_STAR = False or install bash.')

    start_time = time.time()
    bash_executable = shutil.which('bash') or '/bin/bash'

    for command, bam_path, star_tmp_path in zip(star_align_commands, bam_paths, star_tmp_paths):
        bam_path.parent.mkdir(parents=True, exist_ok=True)
        if bam_path.exists() and bam_path.stat().st_size == 0:
            if CLEAN_FAILED_STAR_OUTPUT:
                bam_path.unlink()
                if star_tmp_path.exists():
                    shutil.rmtree(star_tmp_path)
            else:
                raise RuntimeError(
                    f'Previous failed STAR output remains: {bam_path}. '
                    'Set CLEAN_FAILED_STAR_OUTPUT = True once, rerun this cell, then set it back to False.'
                )

        if isinstance(command, str):
            run_command = command.replace('STAR ', shlex.quote(star) + ' ', 1)
            print('Running:', run_command)
            subprocess.run(run_command, check=True, shell=True, executable=bash_executable)
        else:
            run_command = [star] + command[1:]
            print('Running:', ' '.join(run_command))
            subprocess.run(run_command, check=True)

        index_command = [samtools, 'index', str(bam_path)]
        print('Running:', ' '.join(index_command))
        subprocess.run(index_command, check=True)
    print(f'Total STAR alignment time: {time.time() - start_time:.2f} seconds')
else:
    print('Set RUN_STAR_ALIGN = True only after STAR index is ready.')


## STAR alignment結果がそろっているか確認する

各sampleにsorted BAMが存在すれば、featureCountsでgeneごとのread数を数えられる。


In [ ]:
bam_status = pd.DataFrame(
    [
        {
            'sample_id': sample_id,
            'bam_exists': bam_path.exists(),
            'bai_exists': Path(str(bam_path) + '.bai').exists(),
            'bam_path': str(bam_path.relative_to(PROJECT_DIR)),
        }
        for sample_id, bam_path in zip(SAMPLES['sample_id'], bam_paths)
    ]
)
bam_status


## featureCountsコマンドを作る

featureCountsは、BAM中のmapping済みreadとGTF中のgene annotationを重ねて、geneごとのcountを作る。

考え方は次の式である。

$$
\mathrm{count}_{g,s} = \sum_i I(\mathrm{read}_i \text{ overlaps exons of gene } g)
$$

- `g` はgeneである。
- `s` はsampleである。
- `read_i` はBAMファイル中の1つのreadまたはread pairである。
- $I(...)$ は条件が真なら1、偽なら0を返す関数である。

小さい例では、`sample1` の5 readのうち3 readが `TP53` のexonに重なれば、`count_TP53,sample1 = 3` である。

featureCountsの出力は、最初は次のような表になる。

```text
Geneid            Chr   Start  End  Strand  Length  sample1.bam  sample2.bam
ENSG00000141510   chr17  ...    ...  -       19149   120          135
ENSG00000177606   chr1   ...    ...  -       2758    45           50
```

このあとDESeq2向けに、`gene_id` とsample列だけのmatrixへ整形する。


In [ ]:
featurecounts_output = FEATURECOUNTS_DIR / 'featureCounts_gene_counts.txt'
featurecounts_command = [
    'featureCounts',
    '-T', str(THREADS),
    '-p',
    '-s', str(FEATURECOUNTS_STRANDNESS),
    '-a', str(GTF_PATH),
    '-o', str(featurecounts_output),
    *[str(path) for path in bam_paths],
]

print(' '.join(featurecounts_command[:14]), '...')
print('BAM files:', len(bam_paths))
print('Output:', featurecounts_output.relative_to(PROJECT_DIR))


## featureCountsを実行する

初回だけ `RUN_FEATURECOUNTS = True` にする。

`FEATURECOUNTS_STRANDNESS = 0` はunstrandedを意味する。strand-specific libraryだと分かっている場合は、`1` または `2` に変更する。


In [ ]:
RUN_FEATURECOUNTS = True

if RUN_FEATURECOUNTS:
    featurecounts = shutil.which('featureCounts')
    if featurecounts is None:
        raise RuntimeError('featureCounts was not found. Activate the rna-seq environment first.')
    if not GTF_PATH.exists():
        raise FileNotFoundError(GTF_PATH)
    missing_bams = [path for path in bam_paths if not path.exists()]
    if missing_bams:
        raise FileNotFoundError(f'Missing BAM files: {missing_bams[:3]}')

    FEATURECOUNTS_DIR.mkdir(parents=True, exist_ok=True)
    command = [featurecounts] + featurecounts_command[1:]
    print('Running:', ' '.join(command))
    subprocess.run(command, check=True)
else:
    print('Set RUN_FEATURECOUNTS = True only after all sorted BAM files exist.')


## featureCounts結果からgene count matrixを作る

`featureCounts_gene_counts.txt` は説明列を含むので、DESeq2へ渡す前に次の形へ整える。

```text
gene_id            NAC_S2_2h_1  NAC_S2_2h_2  Non_1
ENSG00000141510    120          135          128
ENSG00000177606    45           50           48
```

この表では、行がgene、列がsample、値がそのgeneへ割り当てられたread pair数である。


In [ ]:
BUILD_COUNT_MATRIX_FROM_FEATURECOUNTS = True

if BUILD_COUNT_MATRIX_FROM_FEATURECOUNTS:
    if not featurecounts_output.exists():
        raise FileNotFoundError(featurecounts_output)

    featurecounts_table = pd.read_csv(featurecounts_output, sep='\t', comment='#')
    count_columns = list(featurecounts_table.columns[6:])
    if len(count_columns) != len(SAMPLES):
        raise ValueError(f'Expected {len(SAMPLES)} count columns, found {len(count_columns)}')

    rename_columns = {'Geneid': 'gene_id'}
    rename_columns.update(dict(zip(count_columns, SAMPLES['sample_id'])))

    count_matrix = featurecounts_table[['Geneid', *count_columns]].rename(columns=rename_columns)
    count_matrix[SAMPLES['sample_id']] = count_matrix[SAMPLES['sample_id']].round().astype(int)
    OUT_COUNT_MATRIX.parent.mkdir(parents=True, exist_ok=True)
    count_matrix.to_csv(OUT_COUNT_MATRIX, sep='\t', index=False)

    print('Wrote:', OUT_COUNT_MATRIX.relative_to(PROJECT_DIR))
    display(count_matrix.head())
else:
    print('Set BUILD_COUNT_MATRIX_FROM_FEATURECOUNTS = True after featureCounts output exists.')


## count matrixの形を確認する

count matrixができたら、列名が `samples.tsv` の `sample_id` と一致しているか確認する。


In [ ]:
if OUT_COUNT_MATRIX.exists():
    counts = pd.read_csv(OUT_COUNT_MATRIX, sep='\t')
    print('shape:', counts.shape)
    print('columns:', list(counts.columns[:10]))
    display(counts.head())

    missing_samples = set(SAMPLES['sample_id']) - set(counts.columns)
    extra_samples = set(counts.columns) - {'gene_id'} - set(SAMPLES['sample_id'])
    print('missing_samples:', sorted(missing_samples))
    print('extra_samples:', sorted(extra_samples))
else:
    print('No genome-mapping count matrix yet:', OUT_COUNT_MATRIX.relative_to(PROJECT_DIR))


**よくある間違い**

- transcript FASTAだけを用意して、genome FASTAを用意していない。
- STAR indexを作らずにalignmentへ進む。
- BAMが一部sampleだけ欠けた状態でfeatureCountsを実行する。
- stranded libraryなのに `FEATURECOUNTS_STRANDNESS = 0` のまま数える。

**小さい練習**

`bam_status` で `False` のsampleがあれば、そのsampleのSTARログを確認しよう。まずは1 sampleだけ `RUN_STAR_ALIGN = True` で試し、出力BAM名を確認してから全sampleへ進むと安全である。
